# Why were `Dataset` and `DataLoader` introduced?

Before these classes, developers manually handled:

- loading data
- slicing batches
- shuffling
- preprocessing
- sampling
- batching

This became difficult for:

- large datasets
- image datasets
- text datasets
- parallel loading
- transformations
- mini-batch training

PyTorch introduced:

- `Dataset`
- `DataLoader`

to standardize and automate the data pipeline.


# Problem with Batch Gradient Descent

In Batch Gradient Descent:

```text
Entire dataset → one forward pass → one backward pass → one update
```

Example:

```text
100000 rows processed together
```

Problems:

1. Huge memory consumption
2. Slow training
3. Slow parameter updates
4. Not scalable for large datasets


# Solution → Mini-Batch Gradient Descent

Instead of using all rows together:

```text
100000 rows
```

we split data into smaller batches:

```text
32 rows
64 rows
128 rows
```

Now training becomes:

```text
Batch 1 → update weights
Batch 2 → update weights
Batch 3 → update weights
```

Advantages:

- faster convergence
- lower memory usage
- more frequent updates
- scalable training


# Main Challenges in Manual Mini-Batch Implementation

Without Dataset/DataLoader, developers manually handled:

## 1. Data Loading

Reading:

- CSV files
- images
- folders
- audio
- text


## 2. Batch Creation

Manually slicing:

```python
X[i:i+batch_size]
```


## 3. Shuffling

Randomizing data every epoch manually.


## 4. Data Transformations

Applying:

- normalization
- resizing
- augmentation
- tokenization

No standard place existed.


## 5. Parallel Loading

CPU stayed idle while data loaded slowly from disk.


## 6. Sampling

Needed custom sampling strategies like:

- random sampling
- sequential sampling
- class-balanced sampling


# PyTorch Solution

PyTorch separates responsibilities into:

| Component | Responsibility |
|---|---|
| Dataset | How to access one sample |
| DataLoader | How to create and serve batches |


# What is a Dataset?

A `Dataset` defines:

```text
How to fetch ONE data sample
```

It acts like a bridge between:

```text
Raw Data → PyTorch Model
```

# Main Responsibilities of Dataset

Dataset:

1. Knows where data exists
2. Loads one sample
3. Returns `(X, y)`
4. Applies transformations
5. Supports lazy loading


# Core Methods of Dataset

To create a custom dataset:

```python
class MyDataset(Dataset):
```

you mainly implement 3 methods.


# 1. `__init__()`

Purpose:

```text
Initialize dataset
```

Usually used for:

- loading CSV
- reading file paths
- defining transforms

Example:

```python
def __init__(self):
    self.data = pd.read_csv("data.csv")
```


# 2. `__len__()`

Purpose:

```text
Returns total number of samples
```

Example:

```python
def __len__(self):
    return len(self.data)
```


# 3. `__getitem__(index)`

Most important method.

Purpose:

```text
Returns one sample using index
```

Example:

```python
def __getitem__(self, idx):

    X = self.data.iloc[idx, :-1]
    y = self.data.iloc[idx, -1]

    return X, y
```

# Why `__getitem__()` is important

PyTorch can fetch samples like:

```python
dataset[0]
dataset[1]
dataset[2]
```

Instead of loading all data into memory.

This enables:

- lazy loading
- memory efficiency
- scalable training


# Real Meaning of Dataset

Dataset answers:

```text
"If I give you an index,
can you give me one sample?"
```

Example:

```python
dataset[5]
```

returns:

```python
(image_tensor, label)
```


# Transformations inside Dataset

One major benefit:

Transformations can be applied inside:

```python
__getitem__()
```

Example:

```python
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])
```

Useful for:

- normalization
- resizing
- augmentation
- tokenization


# What is DataLoader?

`DataLoader` works on top of Dataset.

Dataset gives:

```text
one sample
```

DataLoader combines samples into:

```text
batches
```


# Main Responsibilities of DataLoader

DataLoader:

1. Creates batches
2. Shuffles data
3. Loads data efficiently
4. Supports parallel loading
5. Iterates automatically
6. Uses samplers
7. Uses collate functions


# Basic DataLoader Example

```python
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)
```

Now:

```python
for X_batch, y_batch in loader:
```

PyTorch automatically provides batches.


# Why DataLoader is Needed

Without DataLoader:

```python
for i in range(0, len(X), batch_size):

    batch_X = X[i:i+batch_size]
    batch_y = y[i:i+batch_size]
```

Problems:

- manual batching
- manual shuffling
- inefficient loading
- difficult scaling

DataLoader automates everything.


# DataLoader Workflow

```text
Dataset
   ↓
Sampler
   ↓
Batch Creation
   ↓
Collate Function
   ↓
Training Loop
```


# Role of Sampler

Inside DataLoader there is a sampler.

Sampler decides:

```text
Which indices should be selected?
```


# Types of Samplers

## 1. SequentialSampler

Returns:

```text
0,1,2,3,4...
```

Used when order matters.

Example:

- time-series data


## 2. RandomSampler

Returns randomized indices.

Example:

```text
7,2,19,1,5...
```

Improves generalization.


# Custom Samplers

Can be created for:

- class balancing
- weighted sampling
- custom selection logic

Useful in imbalanced datasets.


# Role of Collate Function

After fetching samples:

```python
(sample1, sample2, sample3...)
```

they must be merged into one batch.

This is handled by:

```python
collate_fn
```


# Default Collate Function

Normally:

```python
[(x1,y1), (x2,y2)]
```

becomes:

```python
X_batch
y_batch
```

using tensor stacking.


# Why Custom Collate Function is Needed

Useful when samples have:

- variable lengths
- different shapes

Common in:

- NLP
- speech processing

Example:

```text
Sentences with different lengths
```

Need padding before batching.


# Important DataLoader Parameters

# 1. `batch_size`

Defines:

```text
How many samples per batch
```

Example:

```python
batch_size=32
```


# 2. `shuffle`

Randomizes data every epoch.

```python
shuffle=True
```

Improves model generalization.


# 3. `num_workers`

Defines number of subprocesses.

Example:

```python
num_workers=4
```

Multiple workers load data in parallel.

Benefits:

- faster loading
- reduced bottleneck


# 4. `drop_last`

Suppose:

```text
100 samples
batch_size=32
```

Last batch:

```text
4 samples
```

If:

```python
drop_last=True
```

last incomplete batch is discarded.

Useful for:

- Batch Normalization
- fixed-size batches


# 5. `pin_memory`

Useful during GPU training.

Speeds up:

```text
CPU → GPU transfer
```

# 6. `sampler`

Allows custom sample selection strategy.


# 7. `collate_fn`

Defines how samples merge into batches.


# Parallel Data Loading

One major advantage of DataLoader:

```python
num_workers > 0
```

Workers load batches simultaneously.

Benefits:

- CPU and GPU work together
- GPU waits less
- faster training


# Complete Training Pipeline

```text
Raw Data
   ↓
Dataset
   ↓
DataLoader
   ↓
Mini-Batches
   ↓
Training Loop
   ↓
Forward Pass
   ↓
Loss Calculation
   ↓
Backward Pass
   ↓
Weight Update
```


# Training Loop Structure

Usually:

## Outer Loop

```python
for epoch in range(epochs):
```

Controls epochs.


## Inner Loop

```python
for X_batch, y_batch in loader:
```

Processes mini-batches.


# Manual vs Dataset/DataLoader Approach

| Feature | Manual Approach | Dataset + DataLoader |
|---|---|---|
| Data Loading | Manual | Standardized |
| Batching | Manual slicing | Automatic |
| Shuffling | Manual | Built-in |
| Transformations | Difficult | Easy |
| Parallel Loading | Difficult | Built-in |
| Sampling | Manual | Flexible samplers |
| Collation | Manual | `collate_fn` |
| Scalability | Poor | Excellent |
| Memory Efficiency | Limited | High |


# Biggest Conceptual Understanding

# Dataset

Defines:

```text
How to access ONE sample
```


# DataLoader

Defines:

```text
How to efficiently deliver MANY samples in batches
```

# Simple Analogy

# Dataset = Library

Knows:

- where books are
- how to fetch one book


# DataLoader = Librarian

Knows:

- how many books per batch
- how to shuffle books
- how to bring books efficiently


# Final One-Line Summary

- `Dataset` = defines data access logic
- `DataLoader` = efficiently feeds mini-batches to the model

# Sample Code

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
# Step 1: Create a synthetic classification dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features=2,       # Number of features
    n_informative=2,    # Number of informative features
    n_redundant=0,      # Number of redundant features
    n_classes=2,        # Number of classes
    random_state=42     # For reproducibility
)

In [3]:
# Convert the data to PyTorch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [4]:
from torch.utils.data import Dataset, DataLoader

In [5]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,idx):
    return self.features[idx], self.labels[idx]

In [6]:
dataset = CustomDataset(X,y)

In [7]:
len(dataset)

10

In [8]:
dataset[2]

(tensor([-2.8954,  1.9769]), tensor(0))

In [16]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [17]:
for batch_features , batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print('-'*50)

tensor([[-1.1402, -0.8388],
        [ 1.8997,  0.8344]])
tensor([0, 1])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-0.5872, -1.9717]])
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [-0.7206, -0.9606]])
tensor([1, 0])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [-1.9629, -0.9923]])
tensor([1, 0])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [ 1.0683, -0.9701]])
tensor([1, 1])
--------------------------------------------------


# Full Training Pipeline

In [18]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [20]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

# scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# encoding
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# convert to tensors
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [25]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,idx):
    return self.features[idx], self.labels[idx]

In [29]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [30]:
# Defining the model
import torch.nn as nn

class MySimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(num_features,1),
        nn.Sigmoid()
    )

  def forward(self,features):
    return self.network(features)



In [31]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# define loss function
loss_function = nn.BCELoss()

In [33]:
# Training Pipeline
# define loop
for epoch in range(25):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model(batch_features)

    # loss calculate
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    # clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.177034392952919
Epoch: 2, Loss: 0.1800612211227417
Epoch: 3, Loss: 0.1128542423248291
Epoch: 4, Loss: 0.04365554824471474
Epoch: 5, Loss: 0.033078283071517944
Epoch: 6, Loss: 0.11946648359298706
Epoch: 7, Loss: 0.23768894374370575
Epoch: 8, Loss: 0.14143256843090057
Epoch: 9, Loss: 0.10804480314254761
Epoch: 10, Loss: 0.013732492923736572
Epoch: 11, Loss: 0.11787261068820953
Epoch: 12, Loss: 0.07954565435647964
Epoch: 13, Loss: 0.07888875901699066
Epoch: 14, Loss: 0.07509196549654007
Epoch: 15, Loss: 0.021105263382196426
Epoch: 16, Loss: 0.03099416196346283
Epoch: 17, Loss: 0.01649494841694832
Epoch: 18, Loss: 0.13049688935279846
Epoch: 19, Loss: 0.10082251578569412
Epoch: 20, Loss: 0.017951607704162598
Epoch: 21, Loss: 0.04426180571317673
Epoch: 22, Loss: 0.1062377542257309
Epoch: 23, Loss: 0.06679222732782364
Epoch: 24, Loss: 0.38550761342048645
Epoch: 25, Loss: 0.002450868021696806


In [34]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9488
